# Costco Intro

In [1]:
import requests
import pandas as pd

# Costco
CIK = "0000909832"

SEC_BASE = "https://" + "data.sec.gov"
endpoint = f"{SEC_BASE}/api/xbrl/companyfacts/CIK{CIK}.json"

headers = {
    # Replace this with your actual email.
    # SEC asks automated users to identify themselves.
    "User-Agent": "Eric Analytics Project your_email@example.com"
}

data = requests.get(endpoint, headers=headers).json()

rows = []

for tag, concept in data["facts"]["us-gaap"].items():

    label = concept.get("label")
    description = concept.get("description")

    for unit, observations in concept["units"].items():

        for obs in observations:

            rows.append({
                "tag": tag,
                "label": label,
                "description": description,
                "unit": unit,
                "value": obs.get("val"),
                "start": obs.get("start"),
                "end": obs.get("end"),
                "fiscal_year": obs.get("fy"),
                "fiscal_period": obs.get("fp"),
                "form": obs.get("form"),
                "filed": obs.get("filed")
            })

df = pd.DataFrame(rows)

print(df.shape)
print(df.head())

(24827, 11)
                      tag                      label  \
0  AccountsPayableCurrent  Accounts Payable, Current   
1  AccountsPayableCurrent  Accounts Payable, Current   
2  AccountsPayableCurrent  Accounts Payable, Current   
3  AccountsPayableCurrent  Accounts Payable, Current   
4  AccountsPayableCurrent  Accounts Payable, Current   

                                         description unit         value start  \
0  Carrying value as of the balance sheet date of...  USD  5.450000e+09   NaN   
1  Carrying value as of the balance sheet date of...  USD  5.450000e+09   NaN   
2  Carrying value as of the balance sheet date of...  USD  5.450000e+09   NaN   
3  Carrying value as of the balance sheet date of...  USD  5.450000e+09   NaN   
4  Carrying value as of the balance sheet date of...  USD  6.317000e+09   NaN   

          end  fiscal_year fiscal_period  form       filed  
0  2009-08-30         2010            Q1  10-Q  2009-12-18  
1  2009-08-30         2010            Q2  

In [2]:
print(df["form"].value_counts())

annual = df[
    (df["form"] == "10-K") &
    (df["fiscal_period"] == "FY")
].copy()

print(annual.shape)

form
10-Q    14682
10-K    10145
Name: count, dtype: int64
(9546, 11)


In [3]:
keywords = (
    "Revenue|Sales|Inventory|AccountsReceivable|"
    "AccountsPayable|NetIncome|Assets|Equity|"
    "CostOf|OperatingIncome"
)

interesting = annual[
    annual["tag"].str.contains(
        keywords,
        case=False,
        regex=True
    )
]

interesting[
    ["tag", "label", "end", "value", "unit"]
].sort_values(
    ["tag", "end"]
).head(100)

,tag,label,end,value,unit
3,AccountsPayableCurrent,"Accounts Payable, Current",2009-08-30,5.450000e+09,USD
7,AccountsPayableCurrent,"Accounts Payable, Current",2010-08-29,5.947000e+09,USD
11,AccountsPayableCurrent,"Accounts Payable, Current",2010-08-29,5.947000e+09,USD
15,AccountsPayableCurrent,"Accounts Payable, Current",2011-08-28,6.544000e+09,USD
19,AccountsPayableCurrent,"Accounts Payable, Current",2011-08-28,6.544000e+09,USD
...,...,...,...,...,...
1274,AssetsCurrent,"Assets, Current",2010-08-29,1.170800e+10,USD
1278,AssetsCurrent,"Assets, Current",2011-08-28,1.370600e+10,USD
1282,AssetsCurrent,"Assets, Current",2011-08-28,1.370600e+10,USD
1286,AssetsCurrent,"Assets, Current",2012-09-02,1.352600e+10,USD
